### Ecommerce Generic - Data Engineer Local ETL

# 1 - Extract/Leitura 

In [92]:
import pandas as pd

In [93]:
df = pd.read_csv("../Ecommerce_DBS.csv")
print(f"Total de registros:{len(df)}")
print(f"Total de colunas:{len(df.columns)}")
df.head()

Total de registros:250000
Total de colunas:14


,Customer ID,Purchase Date,Product Category,Product Price,Quantity,Total Purchase Amount,NPS,Customer Age,Gender,Source,Country,State,Latitude,Longituide
0,46251,08/09/2020,Electronics,12,3,740,7,20,Male,Instagram Campign,Canada,Alberta,55.000000,-115.000000
1,46251,05/03/2022,Home,468,4,2739,8,20,Male,Instagram Campign,Canada,Ontario,50.000000,-85.000000
2,46251,23/05/2022,Home,288,2,3196,10,20,Male,SEM,United States,New Mexico,34.840515,-106.248482
3,46251,12/11/2020,Clothing,196,1,3509,3,20,Male,Instagram Campign,Canada,Saskatchewan,55.000000,-106.000000
4,13593,27/11/2020,Home,449,1,3452,3,20,Female,Instagram Campign,United States,California,36.116203,-119.681564


In [94]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Customer ID            250000 non-null  int64  
 1   Purchase Date          250000 non-null  str    
 2   Product Category       250000 non-null  str    
 3   Product Price          250000 non-null  int64  
 4   Quantity               250000 non-null  int64  
 5   Total Purchase Amount  250000 non-null  int64  
 6   NPS                    250000 non-null  int64  
 7   Customer Age           250000 non-null  int64  
 8   Gender                 250000 non-null  str    
 9   Source                 250000 non-null  str    
 10  Country                250000 non-null  str    
 11  State                  250000 non-null  str    
 12  Latitude               250000 non-null  float64
 13  Longituide             250000 non-null  float64
dtypes: float64(2), int64(6), str(6)
memory usage: 2

# 2 - Data Quality

In [95]:
#Conversão para tipo "data" e verificação de nulos
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'], dayfirst=True)

null_counts = df.isnull().sum()
if null_counts.sum() > 0:
    print(null_counts)
else:
    print("Nenhum Nulo")

#Correção do nome de uma coluna
df = df.rename(columns={"Customer Age ": "Customer Age"})
df = df.rename(columns={"Longituide": "Longitude"})

Nenhum Nulo


In [96]:
#Possíveis anomalias/impossíveis


#Negativos impossíveis
anomalias_negativas = df[
    (df['Product Price'] <= 0) |
    (df['Quantity'] <= 0) |
    (df['Total Purchase Amount'] <= 0)
]

#Idades impóssíveis
anomalias_idade = df[(df['Customer Age'] < 0) | (df['Customer Age'] > 100)]

#Duplicados
duplicados = df[df.duplicated()]

#Anomalias Geográficas
anomalias_geo = df[(df['Latitude'] > 90) | (df['Latitude'] < -90) | (df['Longitude'] > 180) | (df['Longitude'] < -180)]


#Visualizando e removendo
print('ANOMALIAS DETECTADAS:')
print(f"Negativos Impossíveis:{len(anomalias_negativas)}")
print(f"Idades Impóssiveis:{len(anomalias_idade)}")
print(f"Geógraficas Impóssiveis:{len(anomalias_geo)}")
print(f"Duplicadas:{len(duplicados)}")


df_limpo = df.copy()

df_limpo = df_limpo.drop(anomalias_negativas.index)
df_limpo = df_limpo.drop(anomalias_idade.index)
df_limpo = df_limpo.drop(duplicados.index)
df_limpo = df_limpo.drop(anomalias_geo.index)


ANOMALIAS DETECTADAS:
Negativos Impossíveis:0
Idades Impóssiveis:0
Geógraficas Impóssiveis:0
Duplicadas:0


# 3 - Transform/Novas Colunas

In [99]:
#Total Purchase Unique
df['Total Purchase Unique'] = df['Product Price'] * df['Quantity']

#Cliente Recorrente
df['Regular Costume'] = df.duplicated(subset='Customer ID', keep=False)

#Dia da Semana
df['Week Day'] = df['Purchase Date'].dt.day_name()

# 4 - Load/Banco de Dados Local(PostgreSQL)

In [108]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()
usuario = os.getenv("DB_USER")
senha = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
porta = os.getenv("DB_PORT")
banco = os.getenv("DB_NAME")


engine = create_engine(f"postgresql+psycopg2://{usuario}:{senha}@{host}:{porta}/{banco}")

df.to_sql("ecommerce", engine, if_exists="replace", index=False)

1000